# 03 — Ingest AIFS initial conditions

Downloads two consecutive 6-hourly ERA5/IFS snapshots (T−6 h and T+0) from
ECMWF Open Data and stores them in the icechunk/Tigris IC repository.

This is CPU-only and can run in parallel across init dates.

In [ ]:
import datetime as dt
import os
import pathlib

import icechunk
from aifs_modal import ingest

In [ ]:
storm_name = "claudia"
init_date_str = "2025110700"  # %Y%m%d%H

storage_bucket = "aifs-modal-unibe"
initial_conditions_prefix = "aifs-ic-ifs"
initial_conditions_branch = "main"

marker_file = "../data/interim/aifs_ic_claudia_2025110700.done"

In [ ]:
start_date = dt.datetime.strptime(str(init_date_str), "%Y%m%d%H").replace(tzinfo=dt.UTC)

print(f"Storm : {storm_name.capitalize()}")
print(f"Init  : {start_date}")
print(f"IC timestamps: {start_date - dt.timedelta(hours=6)}  →  {start_date}")

initial_conditions_storage = icechunk.tigris_storage(
    bucket=storage_bucket,
    prefix=initial_conditions_prefix,
    region=os.getenv("AWS_REGION", None),
    access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
initial_conditions_repo = icechunk.Repository.open_or_create(initial_conditions_storage)

for date in [start_date - dt.timedelta(hours=6), start_date]:
    print(f"  ingesting {date} ...")
    ingest.ensure_date_ingested(
        date, initial_conditions_repo, branch=initial_conditions_branch
    )

print("Initial conditions ready")

In [ ]:
pathlib.Path(marker_file).touch()
print(f"Marker written: {marker_file}")